# **Environment For Using ZOS-API**

1. Versions of Python later than 3.8 have not been tested compatibility by Zemax staffs.

2. The recommended PythonNET version is 2.5.2

# **.NET Framework**


.NET Framework is Moicrosoft's software framework. It's a big collection of libraries and a runtime environment that lets applications (like Zemax OpticStudio) that written in C#, VB.NET, or F# run on Windows.

When a program or application is written for .NET it's compiled not into machine code but into an **intermediate language (IL)**. When you run the program, the .NET Runtime (CLR-Common Language Runtime) translate the intermediate language into machine code on the fly and runs it.

In a word, .NET Framework is what actually runs those applications on Windows. Python itself doesn't depend on .NET. Zemax OpticStudio's GUI and its ZOS-API are both written in C#/.NET, which means Zemax uses .NET Framework as its runtime

.NET Framework is composed of several components including: 

**CLR (Common Language Runtime)**, which runs .NET programs. 

**Class Library**, which is a huge set of prebuilt code for UI, I/O, networking, XML. database, etc.. 

**Languages**, which are C#, Visual Basic .NET, F#, etc.. 

**ASO.NET**, which is a framework for building web applications. 

**WinForms/WPF**, which is a framework for making Windows desktop applications.

Compared to Python, Python's runtime is CPython interpreter, and file type is `.py`. But .NET's file types are `.cs`, `.vb`, `.fs` compiled to `.dll` or `.exe`.

## **Common Language Runtime (CLR)**

CLR is the heart of .NET like Python's interpreter.

The `.exe` or `.dll` contains **CIL (Common Intermediate Language)** code, the CLR reads that CIL and uses **JIT (Just-In-Time)** compilation to native CPU instructions and executes that machine code, managing Memory allocation, Exceptions and Errors, Type Safety and Security, and Threading and interop with Windows.

## **PythonNET**

Zemax's ZOS-API is built in .NET (C#). But Python normally cannot load .NET libraries directly. `pythonnet` provides a bridge, embeding the .NET Common Language Runtime inside Python

In [ ]:
# PythonNET module, loading Common Language Runtime for PythonNET
import clr
# Used to read meomory address from .NET arrays and pass into NUMPY arrays
import ctypes, sys
from System.Runtime.InteropServices import GCHandle, GCHandleType # type: ignore[import]

**clr package**

`clr` is a pythonnet module for loading CLR and adding assembly references (引用). In .NET, compiled code lives in assemblies, ussually files like `something.dll` or `.exe`. An assembly (程序集文件) contains **Types** (classes, structsm enums, interfaces), **Namespaces** (命名空间，logical folders grouping types, e.g., `System.Runtime.InteropServices`), and **Metadata** (versions, public key, dependencies).

To use types from an assembly, the CLR must load that assembly into the process. In `pythonnet`, you do that with

In [3]:
import clr
clr.AddReference("mscorlib") # by simple name (from Global Assembly Cache / known location)
clr.AddReference(r"D:\Zemax 2025\ANSYS Inc\v252\Zemax OpticStudio\ZOSAPI.dll") # by full file path

After `AddReferences(...)` you can import its types: `from <Namespace> import <Types>`

In [7]:
from ZOSAPI import IZOSAPI_Application # type: ignore[import]

Inside a `.dll` there is usually a top-level namespace also called `ZOSAPI`, and sub-namespaces like `ZOSAPI.Tools`, `ZOSAPI.Analysis`, etc.

Editors/type checkers know how Python modules work, but a .NET namespace is not a Python package. `pythonnet` makes it behave like one at runtime, so execution succeeds even if the editor shows "import not resolved". the `# type: ignore[import]` tells the checker to chill.

**sys package**

`sys` tells about the Python interprester itself and how it's currently running. It allows interacting with the Python runtime environment, reading or modifying system paths, modules, and command-line arguments, controlling program exit behavior, and redirecting standard I/O.

In [8]:
import sys

print("Full Python version and build info:", sys.version)
print("Version info tuple:", sys.version_info)
print("Path to Python executable:", sys.executable)
print("List of directories where Python looks for imports:", sys.path)
print("Short platform ID:", sys.platform)
print("List of command-line arguments passed to Python script:", sys.argv)

Full Python version and build info: 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
Version info tuple: sys.version_info(major=3, minor=8, micro=10, releaselevel='final', serial=0)
Path to Python executable: d:\Msc.Photonics\ZemaxAPI\.venv\Scripts\python.exe
List of directories where Python looks for imports: ['D:\\Python 3.8.10\\python38.zip', 'D:\\Python 3.8.10\\DLLs', 'D:\\Python 3.8.10\\lib', 'D:\\Python 3.8.10', 'd:\\Msc.Photonics\\ZemaxAPI\\.venv', '', 'd:\\Msc.Photonics\\ZemaxAPI\\.venv\\lib\\site-packages', 'd:\\Msc.Photonics\\ZemaxAPI\\.venv\\lib\\site-packages\\win32', 'd:\\Msc.Photonics\\ZemaxAPI\\.venv\\lib\\site-packages\\win32\\lib', 'd:\\Msc.Photonics\\ZemaxAPI\\.venv\\lib\\site-packages\\Pythonwin', 'C:\\Windows\\Microsoft.NET\\Framework64\\v4.0.30319\\']
Short platform ID: win32
List of command-line arguments passed to Python script: ['d:\\Msc.Photonics\\ZemaxAPI\\.venv\\lib\\site-packages\\ipykernel_launcher.py', '--f=c:\\Users\\shown\

**System assembly**

On .NET Framework 4.x which pythonnet 2.5.2 uses, the core assemblies like `mscorlib.dll`, `System.dll` `System.Core.dll` are already loaded by the CLR  at startup or are auto-bound.

`pythonnet` can resolve many core namespaces like `System` from those already-loaded assemblies, so `import System` or `from System import ...` works without an explicit `AddReference`. In contrast, non-core libraries like `System.Xml`, or vendor DLL like `ZOSAPI.dll` are not guaranteed to be loaded. For those, you should `AddReference(...)` first.

## **Dynamic-Link Library (DLL)**

A `.dll` is an assembly that contains compiled code, data, and resources that other programs can use while running, just like `.py`.

Reference a `.dll` and import its top-level namespace in Python:

In [11]:
import clr, os, winreg

# determine location of ZOSAPI_NetHelper.dll & add as reference
aKey = winreg.OpenKey(winreg.ConnectRegistry(None, winreg.HKEY_CURRENT_USER), r"Software\Zemax", 0, winreg.KEY_READ)
zemaxData = winreg.QueryValueEx(aKey, 'ZemaxRoot')
NetHelper = os.path.join(os.sep, zemaxData[0], r'ZOS-API\Libraries\ZOSAPI_NetHelper.dll')
winreg.CloseKey(aKey)

# Add a DLL like ZOS-API libraries as a reference
clr.AddReference(NetHelper)

import ZOSAPI_NetHelper # type: ignore[import]

**os package**

`os` is a standard library module in Python that provides an interface between Python code and the operating system. It lets you do things like working with files and directories, reading or setting environment variables, running system commands, getting information about the current process, and handling paths in a platform-independent way. So it acts like a bridge between Python and underlying OS API.

In [18]:
import os

print(os.name) # 'nt' on Windows, 'posix' on Linux/Mac
print(os.getcwd()) # current working directory
print(os.listdir()) # list files in current directory

nt
d:\Msc.Photonics\ZemaxAPI
['.git', '.venv', 'First_try.py', 'Need_to_know.ipynb', 'PythonStandalone_01_new_file_and_quickfocus.py', 'test.ipynb']


**winreg package**

`winreg` is a Python standard-library module that provides access to the Windows Registry, which is a central hierarchical database where Windows stores configuration and system information. `winreg` is an official interface to read or modify registry keys just like "regedit"

Windows Registry stores all kinds of system and application settings, such as paths to installed programs, file-association rules, hardware driver info, environment variables, startup programs, and user preference. It's organized like a tree (hierarchy) of keys and subkeys.

At the top are root keys such as `HKEY_CLASSES_ROOT`, `HKEY_CURRENT_USER`, `HKEY_LOCAL_MACHINE`, `HKEY_USERS`, `HKEY_CURRENT_CONFIG`. Each key can contain subkeys (like folders) and values (like files)

`aKey = winreg.OpenKey(winreg.ConnectRegistry(None, winreg.HKEY_CURRENT_USER), r"Software\Zemax", 0, winreg.KEY_READ)`

`winreg.ConnectRegistry(None, winreg.HKEY_CURRENT_USER)` connects to a registry hive (a root section of the Windows Registry). `None` means connect to the local computer, not a remote computer. `winre.HKEY_CURRENT_USER` means using the current user's registry branch. Finally, it returns a **handle** to that hive.

In [19]:
print(winreg.ConnectRegistry(None, winreg.HKEY_CURRENT_USER))

<PyHKEY:0x0000000000000AE4>


`<PyHKEY:0x0000000000000AE4>` is a Python object wrapping a Windows registry handl, where the `PyHKEY` is a special Python type used by the `winreg` module, which wraps the native Windows HKEY handle. And the `0x0000000000000AE4` is the actual memory address / handle value assigned by Windows for this open registry connection.

`r"Software\Zemax"` is a raw string literal, the `r` before quotes means don't treat `\` as escape characters. This is the **subkey path** under HKCU, which points to `HKEY_CURRENT_USER\Software\Zemax`, where Zemax store user-specific settings, including the installation root path.

`0` is a placeholder for a reserved parameter.

`winreg.KEY_READ` opens the key with read-only access. `aKey` now represents the registry key `HEKY_CURRENT_USER\Software\Zemax`

`winreg.OpenKey(...)` opens the specified key returns a handle (aKey) that you can later access.

In [25]:
print(aKey)

<PyHKEY:0x0000000000000000>


`zemaxData = winreg.QueryValueEx(aKey, 'ZemaxRoot')`

`winreg.QueryValueEx` reads a value (name + data) from the open registry key (aKey). `"ZemaxRoot"` is the name of the registry value to read. This value contains the installation directory of Zemax (e.g., `"C:\Program Files\Zemax opticStudio"`).

`zemaxData` is a tuple `(value, type)`, where `value` is the **data string (installation path)**, `type` is the data type code (e.g., `winreg.REG_SZ` for a string). So `zemaxData[0]` is the actual installation path string.


In [26]:
print(zemaxData[0])
print(os.sep)
print(zemaxData[1])

D:\Msc.Photonics\Semester 3\Lens Design 2\Projects
\
1


`NetHelper = os.path.join(os.sep, zemaxData[0], r'ZOS-API\Libraries\ZOSAPI_NetHelper.dll')`

`os.path.join()` combines multiple path components safely, using the correct OS separator (`\` on Windows). In this case, it combines `os.sep`, which is an OS separator, and `zemaxData[0]`, which is the actual Zemax root path read from the registry, and `r'ZOS-API\Libraries\ZOSAPI_NetHelper.dll'`, which is the relative path from the Zemax root folder to the specific DLL that helps pythonnet locate and configure ZOS-API correctly.

`NetHelper` becomes a full absolute path string, e.g. `"C:\Program Files\Zemax OpticStudio\ZOS-API\Libraries\ZOSAPI_NetHelper.dll"`

In [27]:
print(NetHelper)

D:\Msc.Photonics\Semester 3\Lens Design 2\Projects\ZOS-API\Libraries\ZOSAPI_NetHelper.dll


# **Usage**

**To change surface type, we need to first get an ISurfaceTypesettings and then assign it.**

In [ ]:
SurfaceType_CB = TheLDE.GetSurfaceAt(4).GetSurfaceTypeSettings(ZOSAPI.Editors.LDE.SurfaceType.CoordinateBreak)
TheLDE.GetSurfaceAt(4).ChangeType(SurfaceType_CB)

NameError: name 'TheLDE' is not defined

**To set a solve to a cell in editor, we need to first create a ISolveData and then assign it.**

In [ ]:
 Solve_ChiefNormal = TheLDE.GetSurfaceAt(4).GetSurfaceCell(ZOSAPI.Editors.LDE.SurfaceColumn.Par1).CreateSolveType(ZOSAPI.Editors.SolveType.PickupChiefRay)
TheLDE.GetSurfaceAt(4).GetSurfaceCell(ZOSAPI.Editors.LDE.SurfaceColumn.Par1).SetSolveData(Solve_ChiefNormal)